# Sun & Abraham Interaction-Weighted Event Studies

**Econometrics Notebook Library · v0.1.0**

## Intuition

Estimate dynamic effects *within treatment cohorts* first, then aggregate only after the cohort-specific effects exist. This prevents a single relative-time coefficient from being contaminated by other cohorts' treatment effects.

For cohort $g$ and relative period $e=t-g$, estimate a cohort-specific effect $\delta_{g,e}$. Then form

$$
\theta_e = \sum_{g\in\mathcal G_e} \omega_{g,e}\,\delta_{g,e},
\qquad
\omega_{g,e}\propto P(G_i=g\mid G_i\in\mathcal G_e).
$$

The control group must be untreated at the comparison date; this notebook uses never-treated units to keep the design transparent.

Reference: [Sun & Abraham, Journal of Econometrics (2021)](https://doi.org/10.1016/j.jeconom.2020.09.006).

## Derivation sketch

Start with cohort-by-event-time interactions rather than common event-time dummies:

$$
Y_{it}=\alpha_i+\lambda_t+
\sum_g\sum_{e\neq -1}\delta_{g,e}
1\{G_i=g\}1\{t-g=e\}+u_{it}.
$$

The omitted period $e=-1$ normalizes each cohort's path. The interaction coefficients are then aggregated into an interaction-weighted (IW) event-study estimand. The crucial change is conceptual: **heterogeneity is permitted before aggregation instead of assumed away inside the regression.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4.5)
pd.set_option("display.max_columns", 30)

In [ ]:
from econnotes.core import simulate_staggered_panel, twfe_event_study, sun_abraham_iw

df = simulate_staggered_panel(n_units=260, seed=31)
sa, cells = sun_abraham_iw(df, window=(-3, 4))
twfe = twfe_event_study(df, window=(-3, 4))
truth = (df[df.treated.eq(1)]
         .groupby("event_time", as_index=False)["tau_true"].mean()
         .rename(columns={"tau_true":"truth"}))
sa.merge(truth, on="event_time", how="left")

In [ ]:
plot = sa.merge(truth, on="event_time", how="left").merge(
    twfe[["event_time","estimate"]].rename(columns={"estimate":"twfe"}), on="event_time", how="left")
fig, ax = plt.subplots()
ax.axhline(0, linewidth=1)
ax.plot(plot.event_time, plot.truth, marker="o", label="Truth")
ax.plot(plot.event_time, plot.estimate, marker="o", label="Sun-Abraham IW")
ax.plot(plot.event_time, plot.twfe, marker="o", label="Naive TWFE")
ax.set(xlabel="Event time", ylabel="Effect", title="Interaction-weighting isolates cohort-specific dynamics")
ax.legend();

## Inspect the hidden layer: cohort × event-time cells

A serious event study should make support visible. If only one cohort remains at a long horizon, the “average dynamic effect” is no longer averaging the same cohort composition as at short horizons.

In [ ]:
cells.query("event_time >= 0 and event_time <= 4")[
    ["cohort","event_time","estimate","se","n_cohort"]
].sort_values(["event_time","cohort"]).head(20)

## Common failure

A clean-looking IW plot can still hide changing composition across horizons. Long-run coefficients often rely disproportionately on early adopters. That is an estimand/composition issue, not a standard-error issue.

Also note that the compact standard-error aggregation in this repository is pedagogical: production implementations should retain the covariance across cohort-event coefficients.

## Researcher failure checklist

- Specify the comparison group: never-treated versus last-/not-yet-treated.
- Show how many cohorts contribute to every event time.
- Do not bin endpoints without understanding what treatment effects enter the bins.
- Do not read pre-period interactions as valid tests if anticipation is plausible.
- Use a production implementation (for example, `fixest::sunab`) for publication-grade inference, and verify its weighting choices.